## ___Multi Response Phylogenetic Mixed Models `MR-PMM` using Markov Chain Monte Carlo generalized linear mixed models `MCMCglmm`___
----------------------

In [2]:
# multi response phylogenetic mixed effect models - look up https://benjamin-halliwell.github.io/MR-PMM/MR-PMM_euc_example_analysis.html

In [1]:
set.seed(20206 - 3 - 9)

suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("MCMCglmm")
    library("brms")
})

In [6]:
data <- read.csv("../../data/chapter2/FREDv3subset/collab_fineroots_log_995_species_means_5states_name_matched_with_phylogeny.csv") # log transformed SRL and RD species averages
phylogeny <- ape::read.tree("../../data/chapter2/uphylomaker/collab_fineroots_log_995_species_means_5states.tre") # phylogenetic tree
stopifnot(data$binominal==phylogeny$tip.label) # make sure the binominal names are matched between the trait data and the phylogeny

In [7]:
ape::is.binary(phylogeny) # damn

[1] FALSE

In [8]:
ape::is.ultrametric(phylogeny) # :)

[1] TRUE

In [9]:
phylogeny <- ape::multi2di(phylogeny) # make the phylogeny completely bifurcating by introducing 0 length branches
sum(phylogeny$edge.length == 0) # damn

[1] 91

In [10]:
phylogeny$edge.length[phylogeny$edge.length==0] <- rnorm(n = sum(phylogeny$edge.length == 0), mean = 1e-6, sd = 1e-8) # replace the 0 length edges with random noise
sum(phylogeny$edge.length == 0) # no more 0 length branches

[1] 0

In [8]:
# "By including more diverse species in our phylogeny, we capture deeper splits that represent more meaningful divergences in the genotype and phenotype of extant lineages,
# precisely the effects we intend to model when analysing inter-species data."
# "One consequence of this, is that shallow topology (near the tips) is less informative than deep topology when attempting to infer patterns of phylogenetic niche conservatism, because differences between genera
# are usually more significant than differences between species within genera."

In [9]:
# "For higher taxonomic ranks (e.g. genus), it will usually be possible to derive a unique consensus tree by sampling a single species from each genus and simply pruning the tree to those tips.
# This approach may be problematic for lower taxonomic ranks however, because more closely related species are less likely to be monophyletic with respect to the taxonomic rank in question.
# Even in such cases, we can easily account for this phylogenetic uncertainty by randomly sampling topologies at the specified rank and fitting our models over this sample of trees."

In [10]:
# sample the phylogeny repeatedly, with one randomly chosen species per genera

NUNIQUE_GENERA <- length(unique(data$F01286)) # number of unique genera in our phylogeny
NSAMPLES = 100 # number of times to sample the original phylogeny
sampled_phylogenies <- list() # sampled sub phylogeneies with genera at tips
sampled_vcvs <- list()

for (i in 1:NSAMPLES) {
    sampled_species <- mapply(split.data.frame(data[, c("binominal", "F01286")], ~F01286), FUN = function(df) sample(df$binominal, 1)) # this is a vector of one randomly sampled species per every genera in the phylogeny
    subphylo <- ape::keep.tip(phy = phylogeny, tip = unname(sampled_species), trim.internal = TRUE) # trimmed phylogeny with one randomly sampled species per genus
    stopifnot(length(subphylo$tip.label)==NUNIQUE_GENERA)
                                     
    if(!ape::is.binary(subphylo)) subphylo <- ape::multi2di(subphylo) # make bifucracting if not already
    if(!ape::is.ultrametric(subphylo)) subphylo <- phytools::force.ultrametric(subphylo, method = "extend", message = FALSE) # make ultrametric if not already

    subphylo$tip.label <- gsub(subphylo$tip.label, pattern = "_[a-z]+", replacement = '') # replace the tip labels (binominal names) with genus names
    
    sampled_phylogenies[[i]] <- subphylo
    sampled_vcvs[[i]] <- ape::vcv.phylo(phy = subphylo, corr = TRUE) # computes the expected variances and covariances of a continuous phenotype assuming it evolves under a Brownian motion model
}

In [34]:
# harvest the genus names
length(unique(gsub(phylogeny$tip.label, pattern = "_{1}[a-z]*", replacement = '')))

[1] 519

In [33]:
unique(gsub(phylogeny$tip.label, pattern = "_{1}[a-z]*", replacement = ''))

[1] "Rudbeckia"              "Ratibida"               "Heliopsis"             
  [4] "Liatris"                "Arnica"                 "Hymenoxys"             
  [7] "Helianthus"             "Pappobolus"             "Ambrosia"              
 [10] "Echinacea"              "Eclipta"                "Coreopsis"             
 [13] "Helichrysum"            "Anaphalis"              "Antennaria"            
 [16] "Leontopodium"           "Inula"                  "Haplopappus"           
 [19] "Symphyotrichum"         "Symphyotrichum-angliae" "Heterotheca"           
 [22] "Solidago"               "Gutierrezia"            "Erigeron"              
 [25] "Doellingeria"           "Aster"                  "Olearia"               
 [28] "Artemisia"              "Achillea"               "Leucanthemum"          
 [31] "Crepis"                 "Taraxacum"              "Picris"                
 [34] "Hypochaeris"            "Nabalus"                "Launaea"               
 [37] "Lactuca"                "Pilosella"              "Cichorium"             
 [40] "Tragopogon"             "Vernonia"               "Arctotheca"            
 [43] "Centaurea"              "Saussurea"              "Atractylis"            
 [46] "Adenocaulon"            "Campanula"              "Carpodetus"            
 [49] "Musineon"               "Zizia"                  "Tordylium"             
 [52] "Daucus"                 "Sanicula"               "Astrantia"             
 [55] "Dendropanax"            "Eleutherococcus"        "Macropanax"            
 [58] "Hedera"                 "Schefflera"             "Neopanax"              
 [61] "Pseudopanax"            "Raukaua"                "Pittosporum"           
 [64] "Pennantia"              "Abelia"                 "Lonicera"              
 [67] "Symphoricarpos"         "Viburnum"               "Sambucus"              
 [70] "Quintinia"              "Ilex"                   "Hedeoma"               
 [73] "Monarda"                "Pycnanthemum"           "Clinopodium"           
 [76] "Thymus"                 "Prunella"               "Agastache"             
 [79] "Salvia"                 "Rosmarinus"             "Lavandula"             
 [82] "Teucrium"               "Lamium"                 "Ballota"               
 [85] "Phlomoides"             "Phlomis"                "Sideritis"             
 [88] "Vitex"                  "Orthocarpus"            "Strobilanthes"         
 [91] "Rungia"                 "Handroanthus"           "Tanaecium"             
 [94] "Catalpa"                "Verbena"                "Verbascum"             
 [97] "Veronica"               "Veronicastrum"          "Plantago"              
[100] "Penstemon"              "Olea"                   "Chionanthus"           
[103] "Phillyrea"              "Osmanthus"              "Nestegis"              
[106] "Fraxinus"               "Syringa"                "Jasminum"              
[109] "Galium"                 "Rubia"                  "Saprosma"              
[112] "Serissa"                "Coprosma"               "Psychotria"            
[115] "Catunaregam"            "Aidia"                  "Coffea"                
[118] "Diplospora"             "Hypobathrum"            "Wendlandia"            
[121] "Alseis"                 "Pertusadina"            "Adina"                 
[124] "Chomelia"               "Asclepias"              "Nerium"                
[127] "Trachelospermum"        "Alstonia"               "Gentiana"              
[130] "Solanum"                "Physalis"               "Withania"              
[133] "Merremia"               "Apodytes"               "Arbutus"               
[136] "Actinidia"              "Styrax"                 "Halesia"               
[139] "Alniphyllum"            "Androsace"              "Lysimachia"            
[142] "Ardisia"                "Embelia"                "Myrsine"               
[145] "Rapanea"                "Diospyros"              "Barringtonia"          
[148] "Madhuca"                "P

In [17]:
length(unique(data$F01286))

[1] 516

In [46]:
setdiff(unique(gsub(phylogeny$tip.label, pattern = "_{1}[a-z]+", replacement = '')), unique(data$F01286))

[1] "Symphyotrichum-angliae" "Leucaena_Leucocephala"  "Festuca-bernardii"     
[4] "Laurelia-zelandiae"     "Blechnum-zelandiae"

In [44]:
c('Symphyotrichum-angliae', 'LeucaenaLeucocephala', 'Festuca-bernardii', 'Laurelia-zelandiae', 'Blechnum-zelandiae') %in% data$F01286

[1] FALSE FALSE FALSE FALSE FALSE

In [45]:
c('Symphyotrichum-angliae', 'LeucaenaLeucocephala', 'Festuca-bernardii', 'Laurelia-zelandiae', 'Blechnum-zelandiae') %in% phylogeny$tip.label

[1] FALSE FALSE FALSE FALSE FALSE